In [ ]:
import glob
import torch
import os

path = "../../databases/DPL/outputs/*"
output_folders = glob.glob(path)

def xyz_from_sdf(sdf_p, n_atoms=None):
    with open(sdf_p) as sdf_f:
        lines = [l.split() for l in sdf_f]
    lines = [l for l in lines if len(l) == 16] #only xyz coordinates lines
    xyz = torch.tensor([
        [
            float(l[0]), #x
            float(l[1]), #y
            float(l[2]), #z
        ]
    for l in lines])
    if n_atoms is not None:
        n_mols = int(xyz.shape[0]/n_atoms)
        return xyz.view(n_mols, n_atoms, 3)
    return xyz 

def xyz_from_pdb(pdb_p):
    with open(pdb_p) as pdb_f:
        lines = [l for l in pdb_f if l[:6].strip() == "HETATM"]
    xyz = torch.tensor([
        [
            float(r[30:38].strip()), #x
            float(r[38:46].strip()), #y
            float(r[46:54].strip())  #z
        ] for r in lines
    ])
    return xyz

preds, targets, names = [], [], []
for folder in output_folders:
    name = folder.split("/")[-1]
    pred_f = f"{folder}/sample_ligand.sdf" # this file has all predictions at ounce
    if not os.path.exists(pred_f):
        continue
    target_f = f"../../databases/QBioLiP/nr_lig/{name.replace('_trimed','')}.pdb"
    target = xyz_from_pdb(target_f)
    pred = xyz_from_sdf(pred_f, target.shape[0])
    ranks = torch.tensor([float(r.replace("\n", "")) for r in open(f"{folder}/sample_tmscores.txt").readlines()]) # higher the better
    print(ranks)
    # pred = pred[ranks]

    preds.append(pred)
    targets.append(targets)
    names.append(name)
print(len(names))
print(len(targets))
print(len(preds))